# 2.3 Automated Model Training

Use AutoML to train, tune, and select models with minimal manual intervention.

## Topics
- PyCaret for low-code ML
- H2O AutoML for distributed model training
- AutoGluon for state-of-the-art stacking
- FLAML for fast AutoML
- Model explainability with SHAP


In [ ]:
# !pip install pycaret autogluon flaml h2o shap


In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer(as_frame=True)
df = data.frame
df['target'] = data.target

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print(f'Train: {train_df.shape}, Test: {test_df.shape}')


## Method 1: PyCaret


In [ ]:
from pycaret.classification import setup, compare_models, pull, evaluate_model, save_model

# Setup PyCaret environment
clf = setup(
    data=train_df,
    target='target',
    session_id=42,
    normalize=True,
    fold=5,
    verbose=False
)

# Compare all models
best = compare_models(n_select=3, sort='AUC')
leaderboard = pull()
print(leaderboard[['Model', 'AUC', 'F1', 'Recall', 'Prec.']].head())


## Method 2: FLAML (Fast AutoML)


In [ ]:
from flaml import AutoML
from sklearn.metrics import roc_auc_score

X_train = train_df.drop('target', axis=1)
y_train = train_df['target']
X_test = test_df.drop('target', axis=1)
y_test = test_df['target']

automl = AutoML()
automl.fit(
    X_train, y_train,
    task='classification',
    metric='roc_auc',
    time_budget=60  # seconds
)

y_pred_proba = automl.predict_proba(X_test)[:, 1]
print(f'FLAML best model: {automl.best_estimator}')
print(f'Test AUC: {roc_auc_score(y_test, y_pred_proba):.4f}')


## Model Explainability with SHAP


In [ ]:
import shap
import matplotlib.pyplot as plt

# Explain model predictions with SHAP
explainer = shap.TreeExplainer(automl.model.estimator)
shap_values = explainer.shap_values(X_test)

# Summary plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values[1], X_test, plot_type='bar', show=False)
plt.title('Feature Importance (SHAP values)')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150)
plt.show()


## Key Takeaways
- AutoML removes guesswork from model selection and hyperparameter tuning
- PyCaret provides the richest out-of-the-box comparison and pipeline API
- FLAML is the fastest option for time-constrained scenarios
- Always explain your best model using SHAP before deploying
